In [1]:
import gradio as gr
import os
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [2]:
tickets= {
    "london": {
        "morning": "$799",
        "afternoon": "$849",
        "evening": "$899",
        "night": "$749"
    },
    "paris": {
        "morning": "$899",
        "afternoon": "$949",
        "evening": "$999",
        "night": "$859"
    },
    "tokyo": {
        "morning": "$1400",
        "afternoon": "$1450",
        "evening": "$1500",
        "night": "$1350"
    },
    "berlin": {
        "morning": "$499",
        "afternoon": "$549",
        "evening": "$599",
        "night": "$469"
    },
    "rome": {
        "morning": "$699",
        "afternoon": "$749",
        "evening": "$799",
        "night": "$659"
    },
    "new york": {
        "morning": "$999",
        "afternoon": "$1049",
        "evening": "$1099",
        "night": "$949"
    }
}


In [3]:
def ticket_prices(destination_city,time):
    if  not destination_city:
        return "Invalid"
    keys=destination_city.lower()
    prices=tickets.get(keys)
    if not prices:
       return "Invalid"
    price=prices.get(time.lower())
    if not price:
        return "Invalid"
    return price

In [4]:
ticket_prices("rome","morning")

'$699'

In [5]:
from openai import OpenAI
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [6]:
system_message=""" You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so."""

In [7]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [8]:
price_format_for_llm={
    "name":"ticket_prices",
    "description":"It will give the fare price for destination city for following timeline",
    "parameters":{
            "type":"object",
            "properties": {
                        "destination_city":{ 
                                 "type":"string",
                                  "description":"This is the Destination City for User"
                                },
                         "time":
                                { 
                                 "type":"string",
                                "description":"This is the flight time for User"
                                }

                        },
            "required":["destination_city","time"],
            "additionalProperties": False

}
}

In [9]:
tools=[{"type":"function","function":price_format_for_llm}]
tools

[{'type': 'function',
  'function': {'name': 'ticket_prices',
   'description': 'It will give the fare price for destination city for following timeline',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': 'This is the Destination City for User'},
     'time': {'type': 'string',
      'description': 'This is the flight time for User'}},
    'required': ['destination_city', 'time'],
    'additionalProperties': False}}}]

In [10]:
import json
def tool_manager(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "ticket_prices":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            times=arguments.get('time')
            price_details = ticket_prices(city,times)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses

In [11]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = tool_manager(message)
        messages.append(message)
        messages.extend(responses)
        for item in messages:
            print(item)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content
    

gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
